# Lab 3: Measuring a Search Engine on Scientific Claims

In Labs 1 and 2 you judged search results by reading them. That works for a handful of queries, but it cannot tell you whether one design is better than another across hundreds of queries, or whether a change you made helped. Real search and retrieval-augmented generation (RAG) systems are tested against **benchmarks**: a fixed set of queries whose correct answers are known in advance.

In this lab you will use **SciFact**, a benchmark built by the Allen Institute for AI. It pairs 5,183 abstracts from the medical and biology literature with short scientific claims written by experts, such as *"Aspirin inhibits the production of PGE2."* For each claim, the experts recorded which abstracts provide evidence for or against it. Your job is to find those abstracts and to **measure** how well each search method does.

By the end you should be able to:

- load a standard retrieval benchmark (documents, queries, and relevance judgments) into Chroma;
- compute recall@k, MRR, and nDCG@10, and explain what each one rewards;
- compare semantic, keyword (BM25), and hybrid search with numbers, and explain the differences with specific claims.

| Part | Activity | Time |
| --- | --- | --- |
| Setup | Install packages, load the embedding model | 5 min |
| 1 | Download and explore SciFact | 15 min |
| 2 | Load the abstracts into Chroma | 15 min (mostly waiting) |
| 3 | Measure semantic search | 25 min |
| 4 | Keyword and hybrid search | 20 min |

This lab does not depend on Labs 1 and 2. It uses the same packages and the same embedding model, and it creates its own database folder, `scifact_db`.

Run each cell in order with **Shift+Enter**.

## Setup

The packages are the same as in Labs 1 and 2. If you installed them already in your virtual environment, skip the next cell.

In [ ]:
%pip install "chromadb>=1.0" requests numpy

Load the same local embedding model as before, `all-MiniLM-L6-v2`. If you ran Lab 1 on this computer, it is already downloaded.

In [ ]:
import os, re, gzip, json, math, random, time
from collections import Counter

import numpy as np
import requests
from chromadb.utils import embedding_functions

embed = embedding_functions.DefaultEmbeddingFunction()
print(len(embed(["Aspirin inhibits the production of PGE2"])[0]))   # 384, as in Lab 1

## Part 1: Download and explore SciFact

You will use the copy of SciFact distributed with **BEIR**, a widely used collection of retrieval benchmarks. Every BEIR dataset has the same three parts:

| File | Contents | SciFact size |
| --- | --- | --- |
| `corpus.jsonl` | The documents: an id, a title, and the text (here, an abstract) | 5,183 abstracts |
| `queries.jsonl` | The queries: an id and the text (here, a claim) | about 1,100 claims |
| `test.tsv` | The **relevance judgments**, also called *qrels*: which documents are correct for each query | 300 claims |

Only the 300 claims in `test.tsv` are used for scoring. (BEIR calls them the test set; in the original SciFact release they are the *development* claims, because the official test answers were never published.)

The next cell downloads the three files from Hugging Face into a `scifact_data` folder, a few megabytes in total. If the download fails, see the note after the cell.

In [ ]:
DATA_DIR = "scifact_data"
os.makedirs(DATA_DIR, exist_ok=True)

# BEIR's copy of SciFact on Hugging Face. The corpus and queries are pinned to an
# older revision because the current one holds only Parquet files.
SOURCES = {
    "corpus.jsonl":  "https://huggingface.co/datasets/BeIR/scifact/resolve/984eed826375f18d27936c4a32bf0f8491e3f414/corpus.jsonl.gz",
    "queries.jsonl": "https://huggingface.co/datasets/BeIR/scifact/resolve/984eed826375f18d27936c4a32bf0f8491e3f414/queries.jsonl.gz",
    "test.tsv":      "https://huggingface.co/datasets/BeIR/scifact-qrels/resolve/main/test.tsv",
}

def local_copy(name):
    """Return the path of a local copy of name (plain or gzipped), or None."""
    for candidate in (name, name + ".gz"):
        path = os.path.join(DATA_DIR, candidate)
        if os.path.exists(path):
            return path
    return None

for name, url in SOURCES.items():
    if local_copy(name):
        print(f"Using local copy of {name}")
        continue
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()          # stop on an error instead of saving an error page
    path = os.path.join(DATA_DIR, os.path.basename(url))
    with open(path, "wb") as f:
        f.write(resp.content)
    print(f"Downloaded {os.path.basename(url)} ({len(resp.content):,} bytes)")

**If the download fails**, download [`scifact.zip`](https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip) from the BEIR project and unzip it. Copy `corpus.jsonl`, `queries.jsonl`, and `qrels/test.tsv` into the `scifact_data` folder next to this notebook, then rerun the cell. It uses local copies when they exist, whether or not they are gzipped.

Now read the three files into Python dictionaries.

In [ ]:
def open_text(path):
    if path.endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8")
    return open(path, encoding="utf-8")

def read_jsonl(name):
    with open_text(local_copy(name)) as f:
        return [json.loads(line) for line in f if line.strip()]

# Abstract id -> {"title": ..., "text": ...}
corpus = {d["_id"]: {"title": d["title"], "text": d["text"]} for d in read_jsonl("corpus.jsonl")}

# Claim id -> set of relevant abstract ids
qrels = {}
with open_text(local_copy("test.tsv")) as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) == 3 and parts[2].isdigit() and int(parts[2]) > 0:   # skips the header row
            qrels.setdefault(parts[0], set()).add(parts[1])

# Claim id -> claim text, keeping only the claims that have judgments
claims = {q["_id"]: q["text"] for q in read_jsonl("queries.jsonl") if q["_id"] in qrels}
QIDS = sorted(claims, key=int)       # a fixed order for every experiment

print(f"{len(corpus):,} abstracts")
print(f"{len(claims)} claims with judgments")
print(f"{sum(len(v) for v in qrels.values())} relevant (claim, abstract) pairs")

### Check your work

You should have 5,183 abstracts and 300 claims. Most claims have exactly one relevant abstract; a few have two or more. Every relevant abstract should be in the corpus.

In [ ]:
print("relevant abstracts per claim:", dict(sorted(Counter(len(v) for v in qrels.values()).items())))
print("judged abstracts missing from the corpus:",
      sum(d not in corpus for v in qrels.values() for d in v))     # expect 0

### Look at a claim and its evidence

In [ ]:
def show_example(qid):
    print("CLAIM:", claims[qid])
    for did in qrels[qid]:
        print(f"  RELEVANT [{did}] {corpus[did]['title']}")
        print("   ", corpus[did]["text"][:400], "...")

show_example(QIDS[0])

### Try it: claims versus abstracts

Read three more examples. Compare the wording of each claim with the wording of its abstract. Notice abbreviations, gene and drug names, numbers, and claims phrased in plain language that the abstract states in technical terms. Each of these is a test of a different search method.

Some claims are **contradicted** by their abstract, not supported. They still count as relevant: the abstract is the evidence a fact-checker would need to read.

In [ ]:
random.seed(3)              # change the seed to see other claims
for qid in random.sample(QIDS, 3):
    show_example(qid)
    print()

### How long are the abstracts?

The embedding model reads at most 256 *tokens*, and scientific text uses roughly 1.3 to 1.5 tokens per word because rare terms are split into pieces. Anything past that limit is silently ignored. In Lab 1 you avoided the problem by chunking the text first. Here you will store whole abstracts, measure the result, and test chunking in the optional extension.

In [ ]:
lengths = np.array([len(f"{d['title']} {d['text']}".split()) for d in corpus.values()])
print(f"words per abstract (with title): median {np.median(lengths):.0f}, "
      f"90th percentile {np.percentile(lengths, 90):.0f}, max {lengths.max()}")

LIKELY_CUT = 190            # about 256 tokens
print(f"{(lengths > LIKELY_CUT).mean():.0%} of abstracts are longer than {LIKELY_CUT} words, "
      "so the model probably never sees their ends")

## Part 2: Load the abstracts into Chroma

Each abstract becomes one record. The stored document is the title followed by the abstract, since titles are short and often the most informative part. The metadata keeps the title and the word count.

The helper below creates a cosine collection, as in Lab 1. If the collection already exists with the right number of records, it reuses it. This means you can restart the kernel without waiting for the embedding again. Pass `rebuild=True` to start over.

Embedding 5,183 abstracts takes several minutes on a laptop CPU, longer than Lab 1 because each record is longer.

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="./scifact_db")

def build_collection(name, ids, documents, metadatas, rebuild=False, batch=250):
    """Create a cosine collection holding these records, or reuse a complete one."""
    existing = [c.name for c in client.list_collections()]
    if name in existing and not rebuild:
        col = client.get_collection(name)
        if col.count() == len(ids):
            print(f"Reusing '{name}' ({col.count():,} records). Pass rebuild=True to start over.")
            return col
    if name in existing:
        client.delete_collection(name)
    col = client.create_collection(name, configuration={"hnsw": {"space": "cosine"}})
    start = time.time()
    for i in range(0, len(ids), batch):
        col.add(ids=ids[i:i+batch], documents=documents[i:i+batch], metadatas=metadatas[i:i+batch])
        print(f"  stored {min(i + batch, len(ids)):,} / {len(ids):,}  ({time.time() - start:.0f} s)")
    return col

def with_title(title, text):
    """Put the title in front of the text, adding a period only if the title lacks one."""
    title = title.rstrip()
    return f"{title} {text}" if title.endswith((".", "?", "!")) else f"{title}. {text}"

DOC_IDS = list(corpus)

collection = build_collection(
    "scifact",
    ids=DOC_IDS,
    documents=[with_title(corpus[d]['title'], corpus[d]['text']) for d in DOC_IDS],
    metadatas=[{"title": corpus[d]["title"], "words": int(n)} for d, n in zip(DOC_IDS, lengths)],
)
print(collection.count(), "records stored")

### Check your work

You should have 5,183 records. Look at one.

In [ ]:
rec = collection.get(ids=[DOC_IDS[0]], include=["documents", "metadatas"])
print(rec["metadatas"][0])
print(rec["documents"][0][:300])

## Part 3: Measure semantic search

To evaluate a search method, run every claim through it and keep the ranked list of abstract ids it returns. This set of ranked lists is called a **run**. Then score each ranked list against the relevance judgments and average over the 300 claims.

### Step 1: three ways to score a ranked list

| Metric | Question it answers | Range |
| --- | --- | --- |
| **Recall@k** | What fraction of the relevant abstracts appear anywhere in the top *k*? | 0 to 1 |
| **Reciprocal rank** | How far down is the *first* relevant abstract? Scores 1 for rank 1, 1/2 for rank 2, 1/3 for rank 3, and 0 if it is not in the top 10. Averaged over claims, it is called **MRR** (mean reciprocal rank). | 0 to 1 |
| **nDCG@10** | Are the relevant abstracts near the top? Each relevant abstract earns credit that shrinks with its rank: 1 at rank 1, 0.63 at rank 2, 0.5 at rank 3, and so on (1 / log₂(rank + 1)). The total is divided by the best possible score, so a perfect ranking scores 1. | 0 to 1 |

nDCG@10 is the headline number reported for BEIR benchmarks. Recall@k matters most when another step reads the top *k* results, such as a person scanning a results page or an LLM in a RAG pipeline: a document that is not retrieved cannot be used, whatever its rank.

In [ ]:
def recall_at(ranking, relevant, k):
    return len(set(ranking[:k]) & relevant) / len(relevant)

def reciprocal_rank(ranking, relevant, k=10):
    for rank, did in enumerate(ranking[:k], start=1):
        if did in relevant:
            return 1 / rank
    return 0.0

def ndcg_at(ranking, relevant, k=10):
    dcg = sum(1 / math.log2(rank + 1)
              for rank, did in enumerate(ranking[:k], start=1) if did in relevant)
    ideal = sum(1 / math.log2(rank + 1) for rank in range(1, min(len(relevant), k) + 1))
    return dcg / ideal

# A worked example: two relevant documents, found at ranks 2 and 4
ranking  = ["B", "A", "D", "C", "E"]
relevant = {"A", "C"}
print("recall@3        =", recall_at(ranking, relevant, 3))
print("recall@5        =", recall_at(ranking, relevant, 5))
print("reciprocal rank =", reciprocal_rank(ranking, relevant))
print(f"nDCG@10         = {ndcg_at(ranking, relevant):.3f}")

### Check your work

Work the example by hand before you trust the code. Recall@3 is 1/2 because only A is in the top three. Recall@5 is 1. The reciprocal rank is 1/2 because the first relevant document, A, is at rank 2. For nDCG, the ranking earns 1/log₂(3) + 1/log₂(5) = 0.631 + 0.431 = 1.062. The best possible ranking, with A and C at ranks 1 and 2, earns 1 + 0.631 = 1.631. The ratio is 0.651.

The next cell defines `evaluate()`, which averages the metrics over all 300 claims, and `show_results()`, which prints every method scored so far as a table.

In [ ]:
from IPython.display import Markdown, display

def evaluate(run):
    """Average each metric over all judged claims. A run maps claim id -> ranked abstract ids."""
    return {
        "nDCG@10":    float(np.mean([ndcg_at(run[q], qrels[q]) for q in QIDS])),
        "MRR@10":     float(np.mean([reciprocal_rank(run[q], qrels[q]) for q in QIDS])),
        "Recall@10":  float(np.mean([recall_at(run[q], qrels[q], 10) for q in QIDS])),
        "Recall@100": float(np.mean([recall_at(run[q], qrels[q], 100) for q in QIDS])),
    }

results = {}     # method name -> its scores; every method you try is added here

def show_results():
    metrics = list(next(iter(results.values())))
    lines = ["| Method | " + " | ".join(metrics) + " |",
             "| --- |" + " --- |" * len(metrics)]
    for name, scores in results.items():
        lines.append(f"| {name} | " + " | ".join(f"{scores[m]:.3f}" for m in metrics) + " |")
    display(Markdown("\n".join(lines)))

### Step 2: run all 300 claims

Chroma accepts many query texts in one call. Ask for the top 100 abstracts per claim so you can compute recall@100. This takes under a minute.

In [ ]:
def semantic_run(col, n=100, batch=50):
    """Return (run, distances): the top n abstract ids per claim and their cosine distances."""
    run, dists = {}, {}
    for i in range(0, len(QIDS), batch):
        qs = QIDS[i:i+batch]
        res = col.query(query_texts=[claims[q] for q in qs], n_results=n, include=["distances"])
        for q, ids_, ds in zip(qs, res["ids"], res["distances"]):
            run[q], dists[q] = ids_, ds
    return run, dists

sem_run, sem_dist = semantic_run(collection)
results["Semantic"] = evaluate(sem_run)
show_results()

### Check your work

Published results for this embedding model on SciFact put nDCG@10 a little above 0.6. Yours should be close. A score far below 0.5 usually means the collection was built with the wrong distance setting or the claims and judgments are out of step. Rerun Part 1, then Part 2 with `rebuild=True`.

### Step 3: look at the failures

Averages hide the interesting part. List the claims for which semantic search found **no** relevant abstract in the top 10, and look at a few. A `*` marks a relevant abstract in the top results.

In [ ]:
def rank_of(run, qid, did):
    """1-based rank of abstract did in a run, or None if it was not retrieved."""
    return run[qid].index(did) + 1 if did in run[qid] else None

def show_claim(qid, runs, top=3):
    """Print a claim, where each run ranked its relevant abstracts, and each run's top results."""
    print("CLAIM:", claims[qid])
    for did in qrels[qid]:
        ranks = ", ".join(f"{name} {rank_of(r, qid, did) or '>100'}" for name, r in runs.items())
        print(f"  RELEVANT [{did}] {corpus[did]['title'][:90]}")
        print(f"      rank: {ranks}")
    for name, r in runs.items():
        print(f"  {name}, top {top}:")
        for did in r[qid][:top]:
            mark = "*" if did in qrels[qid] else " "
            print(f"    {mark} [{did}] {corpus[did]['title'][:90]}")
    print()

missed = [q for q in QIDS if recall_at(sem_run[q], qrels[q], 10) == 0]
print(f"{len(missed)} of {len(QIDS)} claims have no relevant abstract in the semantic top 10\n")
for q in missed[:3]:
    show_claim(q, {"Semantic": sem_run})

**Investigate.** For each failure, decide why the right abstract was missed. Is the claim using an abbreviation or a name the abstract spells differently? Does the evidence sit near the end of a long abstract, past the model's 256-token limit? Are the top results about the same topic but a different finding? Look at more of the `missed` list until you can name at least two different kinds of failure.

### Step 4: can a distance cutoff separate good results from bad?

In Lab 1 you were asked whether you could choose a distance cutoff that reliably separates good matches from poor ones. Now you can measure it. Take the top 10 results for every claim and compare the distances of relevant abstracts with the distances of everything else.

In [ ]:
rel_d, other_d = [], []
for q in QIDS:
    for did, dist in zip(sem_run[q][:10], sem_dist[q][:10]):
        (rel_d if did in qrels[q] else other_d).append(dist)

pct = [10, 25, 50, 75, 90]
print(f"{'percentile':14s}" + "".join(f"{p:>8}" for p in pct))
print(f"{'relevant':14s}" + "".join(f"{np.percentile(rel_d, p):8.3f}" for p in pct))
print(f"{'not relevant':14s}" + "".join(f"{np.percentile(other_d, p):8.3f}" for p in pct))

total_relevant = sum(len(v) for v in qrels.values())
print(f"\n{'keep if distance <':>19} {'results kept':>13} {'share relevant':>15} {'relevant kept':>14}")
for cutoff in np.arange(0.20, 0.81, 0.05):
    kept_rel = sum(d < cutoff for d in rel_d)
    kept = kept_rel + sum(d < cutoff for d in other_d)
    share = f"{kept_rel / kept:.2f}" if kept else "-"
    print(f"{cutoff:19.2f} {kept:13,d} {share:>15} {kept_rel / total_relevant:14.2f}")

**Think about it.** The *share relevant* column is the precision of the cutoff; the *relevant kept* column is its recall. Is there a cutoff that keeps most of the relevant abstracts while discarding most of the rest? If a RAG system passed only results below a fixed distance to an LLM, what would it gain and what would it lose?

### Step 5: is the index the problem?

Chroma's HNSW index is **approximate**: it may skip some true nearest neighbors in exchange for speed. Could that explain the misses? Check by doing an exact search with NumPy, comparing each claim against every stored vector, and comparing the two answers.

In [ ]:
stored = collection.get(include=["embeddings"])
E = np.array(stored["embeddings"])
E = E / np.linalg.norm(E, axis=1, keepdims=True)                 # 5,183 x 384, unit length

Q = np.array(embed([claims[q] for q in QIDS]))
Q = Q / np.linalg.norm(Q, axis=1, keepdims=True)                 # 300 x 384, unit length

sims = Q @ E.T                                                    # every cosine similarity
top = np.argsort(-sims, axis=1)[:, :100]
exact_run = {q: [stored["ids"][j] for j in top[i]] for i, q in enumerate(QIDS)}

agree = np.mean([len(set(exact_run[q][:10]) & set(sem_run[q][:10])) / 10 for q in QIDS])
print(f"HNSW returned {agree:.1%} of the exact top-10 results")
print(f"nDCG@10  HNSW (Chroma): {evaluate(sem_run)['nDCG@10']:.3f}")
print(f"nDCG@10  exact (NumPy): {evaluate(exact_run)['nDCG@10']:.3f}")

At this scale, the approximate index returns almost exactly what brute force would. The misses come from the **vectors**, not the index. This is the lesson of Lab 2 again: the representation decides what the database can find.

## Part 4: Keyword and hybrid search

### BM25

Lab 1's keyword search counted how many query words a chunk contained. Production keyword engines, including Elasticsearch, OpenSearch, and the full-text search built into many databases, use a refinement called **BM25**. It adds three ideas:

1. **Rare words count more.** A match on *PGE2* is far more informative than a match on *patients*. Each word is weighted by its *inverse document frequency* (IDF).
2. **Repeats help, with diminishing returns.** The second occurrence of a word adds less than the first. The parameter `k1` controls how quickly the benefit levels off.
3. **Long documents are discounted.** A long abstract matches more words by chance. The parameter `b` controls the length penalty.

BM25 runs on an **inverted index**: for each word, a list of the documents that contain it and how often. It is the same structure as the index at the back of a book, and the core of every keyword search engine. Build one now.

In [ ]:
STOP = {"the", "a", "an", "of", "to", "and", "in", "on", "for", "is", "are", "was", "were",
        "by", "with", "be", "that", "this", "as", "at", "from", "or", "it", "its", "not",
        "no", "than", "which", "we", "our", "these", "has", "have", "can", "but"}

def tokens(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP]

doc_tokens = [tokens(f"{corpus[d]['title']} {corpus[d]['text']}") for d in DOC_IDS]
doc_len = np.array([len(t) for t in doc_tokens])
avg_len = doc_len.mean()
N = len(DOC_IDS)

index = {}       # word -> list of (document position, count in that document)
for pos, toks in enumerate(doc_tokens):
    for word, count in Counter(toks).items():
        index.setdefault(word, []).append((pos, count))

print(f"{len(index):,} distinct words in the inverted index")
print("'aspirin' appears in", len(index.get("aspirin", [])), "abstracts;",
      "'patients' appears in", len(index.get("patients", [])))

In [ ]:
def bm25_search(query, n=100, k1=1.2, b=0.75):
    """Return up to n abstract ids ranked by BM25 score."""
    scores = np.zeros(N)
    for word in set(tokens(query)):
        postings = index.get(word)
        if not postings:
            continue
        idf = math.log(1 + (N - len(postings) + 0.5) / (len(postings) + 0.5))
        for pos, tf in postings:
            scores[pos] += idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * doc_len[pos] / avg_len))
    best = np.argsort(-scores)[:n]
    return [DOC_IDS[j] for j in best if scores[j] > 0]

bm25_run = {q: bm25_search(claims[q]) for q in QIDS}
results["BM25"] = evaluate(bm25_run)
show_results()

### Check your work

On SciFact, BM25 scores about the same as semantic search, and in many published comparisons slightly better. Small embedding models are trained mostly on general web text, while BM25 gets exact matches on rare technical terms for free. If your BM25 score is far below the semantic score, check the tokenizer by running `tokens()` on a claim.

### Compare claim by claim

The averages can be close while the two methods succeed on different claims. Count the claims where each method wins, then look at the clearest wins on each side.

In [ ]:
sem_ndcg  = {q: ndcg_at(sem_run[q], qrels[q]) for q in QIDS}
bm25_ndcg = {q: ndcg_at(bm25_run[q], qrels[q]) for q in QIDS}

sem_wins  = sorted((q for q in QIDS if sem_ndcg[q] > bm25_ndcg[q]), key=lambda q: bm25_ndcg[q] - sem_ndcg[q])
bm25_wins = sorted((q for q in QIDS if bm25_ndcg[q] > sem_ndcg[q]), key=lambda q: sem_ndcg[q] - bm25_ndcg[q])
ties = len(QIDS) - len(sem_wins) - len(bm25_wins)
print(f"Semantic better: {len(sem_wins)}   BM25 better: {len(bm25_wins)}   tied: {ties}\n")

both = {"Semantic": sem_run, "BM25": bm25_run}
print("=" * 30, "SEMANTIC WINS", "=" * 30)
for q in sem_wins[:2]:
    show_claim(q, both)
print("=" * 30, "BM25 WINS", "=" * 34)
for q in bm25_wins[:2]:
    show_claim(q, both)

**Investigate.** For each example, find the words that made the difference. When BM25 wins, is there a rare term, abbreviation, or number shared by the claim and the abstract? When semantic search wins, does the claim describe the finding in different words?

### Hybrid search with reciprocal rank fusion

Lab 1 ended by mentioning **hybrid search**, which runs both methods and merges the results. The two methods' scores cannot be added directly: one is a cosine distance, and the other is a BM25 score with no fixed upper bound. **Reciprocal rank fusion** (RRF) ignores the scores and uses only the ranks. Each abstract earns 1 / (60 + rank) from each list it appears in, and the abstracts are re-sorted by their total. The constant 60 comes from the paper that introduced the method and keeps a single top rank from dominating.

In [ ]:
def rrf(runs, weights=None, k=60, depth=100):
    """Merge several runs by (weighted) reciprocal rank fusion."""
    weights = weights or [1.0] * len(runs)
    fused = {}
    for q in QIDS:
        scores = Counter()
        for run, w in zip(runs, weights):
            for rank, did in enumerate(run[q], start=1):
                scores[did] += w / (k + rank)
        fused[q] = [did for did, _ in scores.most_common(depth)]
    return fused

hybrid_run = rrf([sem_run, bm25_run])
results["Hybrid (RRF)"] = evaluate(hybrid_run)
show_results()

Hybrid search usually beats both methods on most metrics, because each method recovers some of the other's misses.

### Try it: weight the two methods

Give the semantic run weight `w` and BM25 weight `1 - w`, and see how nDCG@10 changes.

In [ ]:
for w in [0.2, 0.35, 0.5, 0.65, 0.8]:
    scores = evaluate(rrf([sem_run, bm25_run], weights=[w, 1 - w]))
    print(f"semantic weight {w:.2f}: nDCG@10 = {scores['nDCG@10']:.3f}")

**Be careful.** If you pick the best weight by looking at these 300 claims and then report its score on the same 300 claims, the result is optimistic. You tuned the system on the test itself. SciFact also has a separate set of training claims (`train.tsv` in the same Hugging Face repository) that you could use to choose settings before testing. Discussion question 5 asks about this.

### Extension (optional): chunk the long abstracts

In Part 1 you found that many abstracts are longer than the model can read. Test the Lab 1 remedy. Split each abstract into chunks of at most 120 words at sentence boundaries, put the title in front of every chunk so each one keeps its context, and store the chunks in a second collection. To rank abstracts, take each abstract's best-scoring chunk.

This collection has more records than the first, so building it takes several more minutes.

In [ ]:
def chunk_text(text, max_words=120):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks, current = [], []
    for s in sentences:
        if current and len(" ".join(current + [s]).split()) > max_words:
            chunks.append(" ".join(current))
            current = []
        current.append(s)
    if current:
        chunks.append(" ".join(current))
    return chunks

chunk_ids, chunk_docs, chunk_metas = [], [], []
for d in DOC_IDS:
    for i, piece in enumerate(chunk_text(corpus[d]["text"])):
        chunk_ids.append(f"{d}-c{i}")
        chunk_docs.append(with_title(corpus[d]['title'], piece))
        chunk_metas.append({"doc_id": d, "chunk": i})
print(f"{len(chunk_ids):,} chunks from {len(DOC_IDS):,} abstracts")

chunk_collection = build_collection("scifact_chunks", chunk_ids, chunk_docs, chunk_metas)

In [ ]:
def chunked_run(col, n=100, fetch=300, batch=50):
    """Rank abstracts by their best chunk. Fetch extra chunks, since one abstract may fill several slots."""
    run = {}
    for i in range(0, len(QIDS), batch):
        qs = QIDS[i:i+batch]
        res = col.query(query_texts=[claims[q] for q in qs], n_results=fetch, include=["metadatas"])
        for q, chunk_meta in zip(qs, res["metadatas"]):
            ranked = list(dict.fromkeys(m["doc_id"] for m in chunk_meta))   # first (best) chunk wins
            run[q] = ranked[:n]
    return run

chunk_run = chunked_run(chunk_collection)
results["Semantic (chunked)"] = evaluate(chunk_run)
results["Hybrid (chunked + BM25)"] = evaluate(rrf([chunk_run, bm25_run]))
show_results()

Did chunking help? Look at a few claims in your `missed` list from Part 3 and check whether the chunked run finds them. In Lab 1, discussion question 4 asked you to predict how chunk size affects search quality. Compare your prediction with this result.

## Deliverables

Submit this notebook (with its outputs, including the final results table) and a one- to two-page write-up answering the questions below.

### Discussion questions

1. Copy your final results table. Which method is best on nDCG@10, and which on Recall@100? If an LLM will read the top five abstracts for each claim and write an answer, which metric matters most, and why?
2. Pick one claim that semantic search found and BM25 missed, and one where the reverse happened. Explain each in terms of the specific words in the claim and the abstract.
3. Using the distance table from Part 3, could you choose a cutoff that separates relevant from irrelevant abstracts? Compare this with your answer to Lab 1's first discussion question.
4. The exact NumPy search and Chroma's HNSW index gave nearly the same results. Where did the errors come from? At what corpus size would you expect the index itself to start costing accuracy, and how would you measure it?
5. Suppose you chose the hybrid weight that scored best on these 300 claims. Why is that score an overestimate of how the system would do on new claims? How would you choose the weight properly?
6. Many SciFact claims are *contradicted* by their relevant abstract, yet retrieval treats them as correct matches. Why is that the right behavior for the retrieval step? What would a complete fact-checking system need to add on top of the vector database?